In [36]:
import pandas as pd
import numpy as np
from scipy.stats import shapiro

In [37]:
df=pd.read_csv('credit_train.csv')
df.head()

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,14dd8831-6af5-400b-83ec-68e61888a048,981165ec-3274-42f5-a3b4-d104041a9ca9,Fully Paid,445412.0,Short Term,709.0,1167493.0,8 years,Home Mortgage,Home Improvements,5214.74,17.2,NaN,6.0,1.0,228190.0,416746.0,1.0,0.0
1,4771cc26-131a-45db-b5aa-537ea4ba5342,2de017a3-2e01-49cb-a581-08169e83be29,Fully Paid,262328.0,Short Term,NaN,NaN,10+ years,Home Mortgage,Debt Consolidation,33295.98,21.1,8.0,35.0,0.0,229976.0,850784.0,0.0,0.0
2,4eed4e6a-aa2f-4c91-8651-ce984ee8fb26,5efb2b2b-bf11-4dfd-a572-3761a2694725,Fully Paid,99999999.0,Short Term,741.0,2231892.0,8 years,Own Home,Debt Consolidation,29200.53,14.9,29.0,18.0,1.0,297996.0,750090.0,0.0,0.0
3,77598f7b-32e7-4e3b-a6e5-06ba0d98fe8a,e777faab-98ae-45af-9a86-7ce5b33b1011,Fully Paid,347666.0,Long Term,721.0,806949.0,3 years,Own Home,Debt Consolidation,8741.90,12.0,NaN,9.0,0.0,256329.0,386958.0,0.0,0.0
4,d4062e70-befa-4995-8643-a0de73938182,81536ad9-5ccf-4eb8-befb-47a4d608658e,Fully Paid,176220.0,Short Term,NaN,NaN,5 years,Rent,Debt Consolidation,20639.70,6.1,NaN,15.0,0.0,253460.0,427174.0,0.0,0.0


In [38]:
df.drop(columns=['Loan ID','Customer ID',],axis=1,inplace=True)
df.loc[df['Purpose']=='other', 'Purpose'] = 'Other'
df.head()

,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,Fully Paid,445412.0,Short Term,709.0,1167493.0,8 years,Home Mortgage,Home Improvements,5214.74,17.2,NaN,6.0,1.0,228190.0,416746.0,1.0,0.0
1,Fully Paid,262328.0,Short Term,NaN,NaN,10+ years,Home Mortgage,Debt Consolidation,33295.98,21.1,8.0,35.0,0.0,229976.0,850784.0,0.0,0.0
2,Fully Paid,99999999.0,Short Term,741.0,2231892.0,8 years,Own Home,Debt Consolidation,29200.53,14.9,29.0,18.0,1.0,297996.0,750090.0,0.0,0.0
3,Fully Paid,347666.0,Long Term,721.0,806949.0,3 years,Own Home,Debt Consolidation,8741.90,12.0,NaN,9.0,0.0,256329.0,386958.0,0.0,0.0
4,Fully Paid,176220.0,Short Term,NaN,NaN,5 years,Rent,Debt Consolidation,20639.70,6.1,NaN,15.0,0.0,253460.0,427174.0,0.0,0.0


WARTOŚCI PUSTE

In [39]:
df = df.dropna(subset=['Years in current job'])

df['Months since last delinquent'] = df['Months since last delinquent'].fillna(0)
df = df.dropna(subset=['Bankruptcies'])
df = df.dropna(subset=['Tax Liens'])
df = df.dropna(subset=['Maximum Open Credit'])






df.isnull().sum()

Loan Status                         0
Current Loan Amount                 0
Term                                0
Credit Score                    18301
Annual Income                   18301
Years in current job                0
Home Ownership                      0
Purpose                             0
Monthly Debt                        0
Years of Credit History             0
Months since last delinquent        0
Number of Open Accounts             0
Number of Credit Problems           0
Current Credit Balance              0
Maximum Open Credit                 0
Bankruptcies                        0
Tax Liens                           0
dtype: int64

WARTOSCI KATEGORYCZNE

In [40]:
df = pd.get_dummies(data=df, columns=['Term', 'Years in current job', 'Home Ownership', 'Purpose'])

WARTOSCI ODSTAJACE

In [41]:
median_value = df['Annual Income'].median() 
df['Annual Income']=df['Annual Income'].fillna(value=median_value)
median_value2 = df['Current Loan Amount'].median() 
df['Current Loan Amount']=df['Current Loan Amount'].fillna(value=median_value2)
median_value3 = df['Credit Score'].median() 
df['Credit Score']=df['Credit Score'].fillna(value=median_value3)

In [42]:


df['Current Loan Amount']=df['Current Loan Amount'].replace(99999999.0,789250.0)#zastapiane maksimum z tej kolumny
df.loc[df['Credit Score'] > 5000, 'Credit Score'] = 1071.75


In [43]:
def outliers_iqr(df, kolumny):  
    
    mask = pd.Series(True, index=df.index)

 
    for kol in kolumny:
       
        Q1 = df[kol].quantile(0.25)
        Q3 = df[kol].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 2.5 * IQR
        upper_bound = Q3 + 2.5 * IQR

      
        column_mask = (df[kol] >= lower_bound) & (df[kol] <= upper_bound)
        
        
        mask &= column_mask

    
    df_clean = df[mask]
    
    return df_clean

df_clean = outliers_iqr(df, ['Annual Income', 'Current Loan Amount', 'Maximum Open Credit'])

In [44]:
df_clean.count()

Loan Status                       90262
Current Loan Amount               90262
Credit Score                      90262
Annual Income                     90262
Monthly Debt                      90262
Years of Credit History           90262
Months since last delinquent      90262
Number of Open Accounts           90262
Number of Credit Problems         90262
Current Credit Balance            90262
Maximum Open Credit               90262
Bankruptcies                      90262
Tax Liens                         90262
Term_Long Term                    90262
Term_Short Term                   90262
Years in current job_1 year       90262
Years in current job_10+ years    90262
Years in current job_2 years      90262
Years in current job_3 years      90262
Years in current job_4 years      90262
Years in current job_5 years      90262
Years in current job_6 years      90262
Years in current job_7 years      90262
Years in current job_8 years      90262
Years in current job_9 years      90262


TESTY NORMALNOSCI DANYCH

In [45]:
def normality(df, kolumny):
    wynik = []
    for kol in kolumny:
        statystyka, p_value = shapiro(df[kol])  
        wynik.append((kol, statystyka, p_value))  
    return wynik


wynik = normality(df_clean, ['Annual Income', 'Current Loan Amount', 'Credit Score', 'Monthly Debt', 'Number of Open Accounts', 'Maximum Open Credit'])
for kolumna, statystyka, p_value in wynik:
    print(f"Kolumna: {kolumna}, Statystyka: {statystyka:.4f}, P-value: {p_value:.4f}")

Kolumna: Annual Income, Statystyka: 0.9359, P-value: 0.0000
Kolumna: Current Loan Amount, Statystyka: 0.9017, P-value: 0.0000
Kolumna: Credit Score, Statystyka: 0.4896, P-value: 0.0000
Kolumna: Monthly Debt, Statystyka: 0.9369, P-value: 0.0000
Kolumna: Number of Open Accounts, Statystyka: 0.9390, P-value: 0.0000
Kolumna: Maximum Open Credit, Statystyka: 0.9024, P-value: 0.0000


c:\Users\Marceli\anaconda3\Lib\site-packages\scipy\stats\_axis_nan_policy.py:531: UserWarning: scipy.stats.shapiro: For N > 5000, computed p-value may not be accurate. Current N is 90262.
  res = hypotest_fun_out(*samples, **kwds)


In [46]:
df_clean.head()

,Loan Status,Current Loan Amount,Credit Score,Annual Income,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,...,Purpose_Home Improvements,Purpose_Medical Bills,Purpose_Other,Purpose_Take a Trip,Purpose_major_purchase,Purpose_moving,Purpose_renewable_energy,Purpose_small_business,Purpose_vacation,Purpose_wedding
0,Fully Paid,445412.0,709.0,1167493.0,5214.74,17.2,0.0,6.0,1.0,228190.0,...,True,False,False,False,False,False,False,False,False,False
1,Fully Paid,262328.0,724.0,1202263.0,33295.98,21.1,8.0,35.0,0.0,229976.0,...,False,False,False,False,False,False,False,False,False,False
2,Fully Paid,789250.0,741.0,2231892.0,29200.53,14.9,29.0,18.0,1.0,297996.0,...,False,False,False,False,False,False,False,False,False,False
3,Fully Paid,347666.0,721.0,806949.0,8741.90,12.0,0.0,9.0,0.0,256329.0,...,False,False,False,False,False,False,False,False,False,False
4,Fully Paid,176220.0,724.0,1202263.0,20639.70,6.1,0.0,15.0,0.0,253460.0,...,False,False,False,False,False,False,False,False,False,False


EKSPORT DANYCH

In [47]:
df_clean.to_csv('credit_train_clean.csv',index=False)